# Matching c14_master_v08.xlsx site identifiers to SEAD_staging tbl_sites

The xlsx has `site_id` and `raa_id` columns. We match each of them separately against `tbl_sites.national_site_identifier` in the SEAD_staging database.


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


## Load c14_master_v08.xlsx


In [2]:
excel_path = "../data/c14_master_v08.xlsx"
df = pd.read_excel(excel_path)
df[['site_id', 'raa_id']].head()


,site_id,raa_id
0,L1969:5267,Jörlanda 158
1,L1970:9431,Jörlanda 185
2,L1970:9431,Jörlanda 185
3,L1970:9431,Jörlanda 185
4,L1970:9431,Jörlanda 185


### site_id and raa_id overview


In [3]:
print(f"{df['site_id'].notna().sum()} rows with site_id, {df['site_id'].nunique()} distinct values")
print(f"{df['raa_id'].notna().sum()} rows with raa_id, {df['raa_id'].nunique()} distinct values")


29963 rows with site_id, 6118 distinct values
28460 rows with raa_id, 5437 distinct values


## Connect to sead_staging database


In [4]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.environ["DB_PORT"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


## Load tbl_sites


In [5]:
sites = pd.read_sql(
    'select site_id, national_site_identifier, site_name from public.tbl_sites order by site_id',
    engine,
)
print(f'{len(sites)} rows in tbl_sites')
print(f"{sites['national_site_identifier'].notna().sum()} rows with national_site_identifier, "
      f"{sites['national_site_identifier'].nunique()} distinct values")
sites.head()


3462 rows in tbl_sites
1500 rows with national_site_identifier, 1440 distinct values


,site_id,national_site_identifier,site_name
0,1,Slöinge 114,Slöinge Raä 114
1,2,Skrea 194:1,Skrea Raä 194
2,3,Skrea 177:1,Skrea Raä 177
3,4,Onsala Raä 327:1,Onsala 327
4,5,Vallda 293:1,Vallda Raä 293


## Match site_id (xlsx) against national_site_identifier (tbl_sites)


In [6]:
site_id_matches = (
    df[['site_id']].dropna().drop_duplicates()
    .merge(
        sites.rename(columns={'site_id': 'sead_site_id'}),
        left_on='site_id',
        right_on='national_site_identifier',
        how='inner',
    )
)
print(f"{site_id_matches['site_id'].nunique()} of {df['site_id'].nunique()} distinct site_id values "
      "matched a national_site_identifier")
site_id_matches


42 of 6118 distinct site_id values matched a national_site_identifier


,site_id,sead_site_id,national_site_identifier,site_name
0,L1991:2350,3837,L1991:2350,Åby
1,L1988:2436,6382,L1988:2436,Helsingborg 42:1
2,L1989:8982,6392,L1989:8982,Helsingborg 226:1
3,L1983:2662,6376,L1983:2662,Åker 270:1
4,L2011:1331,6412,L2011:1331,Motala 173:1
5,L2015:2168,6421,L2015:2168,Sigtuna 195:1
6,L1959:7102,6363,L1959:7102,Algutsboda 79:1
7,L2017:1568,6424,L2017:1568,Adelsö 119:1
8,L1989:3865,5790,L1989:3865,Falsterbo kyrka
9,L1989:3865,6390,L1989:3865,Falsterbo 15:1


## Match raa_id (xlsx) against national_site_identifier (tbl_sites)


In [7]:
raa_id_matches = (
    df[['raa_id']].dropna().drop_duplicates()
    .merge(
        sites.rename(columns={'site_id': 'sead_site_id'}),
        left_on='raa_id',
        right_on='national_site_identifier',
        how='inner',
    )
)
print(f"{raa_id_matches['raa_id'].nunique()} of {df['raa_id'].nunique()} distinct raa_id values "
      "matched a national_site_identifier")
raa_id_matches


106 of 5437 distinct raa_id values matched a national_site_identifier


,raa_id,sead_site_id,national_site_identifier,site_name
0,Odensala 6,329,Odensala 6,Odensala 6 (386)
1,Strängnäs 266,3580,Strängnäs 266,Lunda omr. B
2,Sigtuna 195,3463,Sigtuna 195,St. Gatan. Kv. Handelsmannen 8-9
3,Sigtuna 195,3615,Sigtuna 195,kv. Trädgårdsmästaren
4,Sigtuna 195,3628,Sigtuna 195,kv Trädgårdsmästaren
...,...,...,...,...
118,Norra Nöbbelöv 13,3540,Norra Nöbbelöv 13,Norra Nöbbelöv
119,Grevie 363,3722,Grevie 363,Grevie 363
120,Linköping 188,3585,Linköping 188,Linköping 188
121,Norrsunda 167,3538,Norrsunda 167,Norrsunda 167
